# Ranking de cenários (Entrega 2)

Reúne os 10 melhores cenários da planilha de Contabilidade

- **Entrada:** `src/Staging/kpis.parquet`, gerado por `kpis.ipynb`.
- **Saída:** `src/Staging/top10_referencia.csv`.


## Bloco 1 - Top 10 da planilha

Posição, cenário e score final dos 10 melhores cenários, copiados da aba `ranking_cenarios`.

In [1]:
from pathlib import Path

import pandas as pd

PASTA_STAGING = Path("../Staging")

top10 = pd.DataFrame(
    [
        (1, "Cen_01128", 0.8789375),
        (2, "Cen_01127", 0.865095),
        (3, "Cen_01126", 0.831432),
        (4, "Cen_01088", 0.81385),
        (5, "Cen_01087", 0.805875),
        (6, "Cen_01086", 0.800575),
        (7, "Cen_01125", 0.797818),
        (8, "Cen_01124", 0.791987),
        (9, "Cen_01085", 0.78525),
        (10, "Cen_00736", 0.7777),
    ],
    columns=["posicao", "cenario", "score"],
)

top10

,posicao,cenario,score
0,1,Cen_01128,0.878938
1,2,Cen_01127,0.865095
2,3,Cen_01126,0.831432
3,4,Cen_01088,0.813850
4,5,Cen_01087,0.805875
5,6,Cen_01086,0.800575
6,7,Cen_01125,0.797818
7,8,Cen_01124,0.791987
8,9,Cen_01085,0.785250
9,10,Cen_00736,0.777700


## Bloco 2 - Tabela de referência (120 linhas)

Junta o Top 10 ao `kpis.parquet` (10 cenários x 12 anos) e grava `top10_referencia.csv` com os 22 KPIs das abas, na mesma ordem em que aparecem na planilha:

- **Aba 10 - Fluxo de caixa (método direto):** as 10 linhas `dfc_*`, de recebimentos a caixa final.
- **Aba 8 - Índices da DFC:** `taxa_conversao`, `fcl`, `fco_ebitda`, `fco_lucro_liquido`, `capex_receita`, `capex_da`, `cobertura_juros_caixa`, `payout_caixa` e `cobertura_dividendos`.
- **Aba 9 - Capital de giro:** `pmr`, `pmp` e `ciclo_financeiro` (PMP e ciclo ficam vazios no Ano 12, onde a conta de fornecedores não existe).

Os valores em R$ são divididos por 1 milhão e ganham o sufixo `_mi`. A última linha mostra o cenário em 1º lugar, com os anos nas colunas.

In [2]:
kpis = pd.read_parquet(PASTA_STAGING / "kpis.parquet")

fluxo_de_caixa = [
    "dfc_recebimentos",
    "dfc_pagamentos",
    "dfc_fco",
    "dfc_investimentos",
    "dfc_juros_liquidos",
    "dfc_dividendos",
    "dfc_financiamento",
    "dfc_variacao_liquida",
    "dfc_caixa_inicial",
    "dfc_caixa_final",
]
indices_dfc = [
    "taxa_conversao",
    "fcl",
    "fco_ebitda",
    "fco_lucro_liquido",
    "capex_receita",
    "capex_da",
    "cobertura_juros_caixa",
    "payout_caixa",
    "cobertura_dividendos",
]
capital_de_giro = ["pmr", "pmp", "ciclo_financeiro"]

colunas_em_reais = fluxo_de_caixa + ["fcl"]
colunas_razao = [
    "taxa_conversao",
    "fco_ebitda",
    "fco_lucro_liquido",
    "capex_receita",
    "capex_da",
    "cobertura_juros_caixa",
    "payout_caixa",
    "cobertura_dividendos",
]

referencia = top10.merge(kpis, on="cenario").sort_values(["posicao", "ano_num"])

for coluna in colunas_em_reais:
    referencia[coluna + "_mi"] = (referencia[coluna] / 1e6).round(3)
for coluna in colunas_razao:
    referencia[coluna] = referencia[coluna].round(4)
for coluna in capital_de_giro:
    referencia[coluna] = referencia[coluna].round(2)

nomes_no_arquivo = []
for coluna in fluxo_de_caixa + indices_dfc + capital_de_giro:
    if coluna in colunas_em_reais:
        nomes_no_arquivo.append(coluna + "_mi")
    else:
        nomes_no_arquivo.append(coluna)

colunas_finais = ["posicao", "cenario", "score", "ano_calendario"] + nomes_no_arquivo
referencia = referencia[colunas_finais]

referencia.to_csv(PASTA_STAGING / "top10_referencia.csv", index=False)

print("Tabela de referência:", referencia.shape, "-> top10_referencia.csv")

primeiro = referencia[referencia["posicao"] == 1]
primeiro.set_index("ano_calendario").drop(columns=["posicao", "cenario", "score"]).T

Tabela de referência: (120, 26) -> top10_referencia.csv


ano_calendario,2027,2028,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038
dfc_recebimentos_mi,5541.4810,6180.8520,6620.1870,7102.9570,7637.9850,8227.8930,8884.9810,9608.6160,10316.6420,11132.6740,12012.3560,15821.8410
dfc_pagamentos_mi,-2177.9670,-2669.7460,-2823.6360,-3359.1390,-3857.3410,-3665.3130,-3890.6260,-4217.1740,-4927.4880,-5563.3200,-5326.7310,-6714.1550
dfc_fco_mi,3363.5140,3511.1060,3796.5520,3743.8180,3780.6440,4562.5800,4994.3550,5391.4420,5389.1540,5569.3540,6685.6250,9107.6860
dfc_investimentos_mi,-961.6340,-571.1520,-243.9410,-136.4430,-67.6930,-131.1090,-73.5950,-46.2890,-183.2200,-67.5920,-51.3720,-8.9010
dfc_juros_liquidos_mi,-836.3100,-712.5650,-611.6620,-1779.1250,-1652.5040,-1150.6710,-1066.5870,-381.3720,-359.9970,-338.1010,-315.3010,-294.5730
dfc_dividendos_mi,-1532.4080,-2553.7460,-2069.7380,-2421.0620,-2460.2960,-2760.5690,-2990.1420,-3200.5860,-3451.7630,-3798.6120,-4256.4700,-15879.8130
dfc_financiamento_mi,-2368.7180,-3266.3110,-2681.4000,-4200.1870,-4112.8000,-3911.2400,-4056.7290,-3581.9580,-3811.7600,-4136.7120,-4571.7710,-16174.3860
dfc_variacao_liquida_mi,33.1610,-326.3580,871.2110,-592.8130,-399.8490,520.2300,864.0310,1763.1960,1394.1740,1365.0490,2062.4810,-7075.6000
dfc_caixa_inicial_mi,774.6150,807.7760,481.4180,1352.6290,759.8170,359.9670,880.1980,1744.2290,3507.4250,4901.5980,6266.6480,8329.1290
dfc_caixa_final_mi,807.7760,481.4180,1352.6290,759.8170,359.9670,880.1980,1744.2290,3507.4250,4901.5980,6266.6480,8329.1290,1253.5290
